# Lab 1: Framework

Show the 25 non-stopwords with the highest number of occurrences in the file `blake-poems.txt` of the Gutenberg corpus.

In [1]:
from pathlib import Path

# The notebook runs from labs/, so the workspace root is the parent folder.
ROOT = Path.cwd().parent

NLTK_DIR = ROOT / "models" / "nltk_data"

# Load the necessary libraries and download required datasets
import nltk
from nltk.corpus import gutenberg, stopwords
from collections import Counter

# NLTK packages needed by this notebook.
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("gutenberg", download_dir=NLTK_DIR)
nltk.download("stopwords", download_dir=NLTK_DIR)

[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# Read the text of William Blake's poems from the Gutenberg corpus
tokens = gutenberg.words('blake-poems.txt')
print(f"Tokens in blake-poems.txt: {len(tokens)}")
print(f"Title: {' '.join(tokens[:18])}")

Tokens in blake-poems.txt: 8354
Title: [ Poems by William Blake 1789 ] SONGS OF INNOCENCE AND OF EXPERIENCE and THE BOOK of THEL


In [3]:
# Read the stop words from the NLTK stopwords corpus
stop_words = set(stopwords.words('english'))
print(f"English stop words: {len(stop_words)}")

English stop words: 198


In [4]:
# Filter the words to remove stop words and non-alphabetic tokens, and convert them to lowercase
words = [
    token.lower()
    for token in tokens
    if token.isalpha() and token.lower() not in stop_words
]
print(f"Non-stopwords: {len(words)}")

Non-stopwords: 3805


In [5]:
# Count the frequency of each word in the filtered list
word_counts = Counter(words)
print(f"Distinct non-stopwords: {len(word_counts)}")

Distinct non-stopwords: 1400


In [6]:
# Display the 25 most common words in the filtered list
for rank, (word, count) in enumerate(word_counts.most_common(25), start=1):
    print(f"{rank:>2}. {word:<8} {count}")

 1. little   45
 2. thee     42
 3. like     35
 4. thou     35
 5. thy      31
 6. love     29
 7. sweet    28
 8. night    28
 9. joy      25
10. away     24
11. weep     24
12. father   22
13. sleep    21
14. happy    19
15. shall    19
16. day      19
17. mother   19
18. child    18
19. every    17
20. never    17
21. thel     16
22. hear     16
23. green    16
24. er       16
25. voice    16


In [7]:
# 1. Ties: several words share the count of the 25th word, so the cut-off at 25 is partly arbitrary.
cutoff = word_counts.most_common(25)[-1][1]
tied = [word for word, count in word_counts.items() if count == cutoff]
print(f"Words with {cutoff} occurrences: {tied}")

# 2. 'er' is not a word: gutenberg.words() splits contractions such as "o'er" into 'o', "'" and 'er'.
contexts = [' '.join(tokens[i - 2:i + 2]) for i, token in enumerate(tokens) if token.lower() == 'er']
print(f"\nOccurrences of 'er': {len(contexts)}, e.g. {contexts[:4]}")
print(f"'o' in stop list: {'o' in stop_words}, 'er' in stop list: {'er' in stop_words}")

# 3. The NLTK stop list is for modern English, so archaic pronouns and verb forms are kept.
archaic = ['thee', 'thou', 'thy', 'thine', 'art', 'doth', 'dost', 'hath', 'shalt', 'wilt']
print(f"\nArchaic forms not in the stop list: { {word: word_counts[word] for word in archaic} }")
print(f"Archaic forms in the stop list: {[word for word in archaic if word in stop_words]}")
print(f"Modern equivalents in the stop list: { {word: word in stop_words for word in ['you', 'your', 'yours']} }")

Words with 16 occurrences: ['thel', 'hear', 'green', 'er', 'voice', 'infant', 'see', 'human']

Occurrences of 'er': 16, e.g. ["o ' er the", "O ' er my", "o ' er my", "o ' er thee"]
'o' in stop list: True, 'er' in stop list: False

Archaic forms not in the stop list: {'thee': 42, 'thou': 35, 'thy': 31, 'thine': 2, 'art': 6, 'doth': 7, 'dost': 5, 'hath': 1, 'shalt': 2, 'wilt': 3}
Archaic forms in the stop list: []
Modern equivalents in the stop list: {'you': True, 'your': True, 'yours': True}


In [8]:
# Top 25 again, adding the archaic forms and the 'er' fragment to the stop list.
extended_stop_words = stop_words | set(archaic) | {'er'}
extended_counts = Counter(word for word in words if word not in extended_stop_words)

for rank, (word, count) in enumerate(extended_counts.most_common(25), start=1):
    print(f"{rank:>2}. {word:<8} {count}")

 1. little   45
 2. like     35
 3. love     29
 4. sweet    28
 5. night    28
 6. joy      25
 7. away     24
 8. weep     24
 9. father   22
10. sleep    21
11. happy    19
12. shall    19
13. day      19
14. mother   19
15. child    18
16. every    17
17. never    17
18. thel     16
19. hear     16
20. green    16
21. voice    16
22. infant   16
23. see      16
24. human    16
25. cloud    15


## Discussion

**Method.** The 8354 tokens of `blake-poems.txt` are lowercased, the non-alphabetic tokens (punctuation, numbers) and the 198 NLTK English stop words are removed, and the remaining 3805 words (1400 distinct) are counted.

**Result.** The list reflects the main themes of *Songs of Innocence and of Experience*: childhood and family (*little*, *child*, *mother*, *father*), emotions (*love*, *joy*, *weep*, *happy*) and nature (*night*, *sleep*, *green*, *day*). *Thel* is a proper noun, the main character of *The Book of Thel*, which is also in the file.

However, some entries are not meaningful words:

- **Archaic forms.** *thee* (42), *thou* (35) and *thy* (31) are in positions 2, 4 and 5. They are personal pronouns and determiners, the same kind of words that the stop list removes in modern English (*you*, *your*), but the NLTK list does not include them. Other archaic forms (*doth*, *art*, *dost*, *wilt*…) are less frequent and do not reach the top 25.
- **Tokenization artifact.** *er* (16) is not a word. `gutenberg.words()` splits *o'er* into *o*, *'* and *er*; *o* is in the stop list, but *er* is not.
- **Ties at the cut-off.** 8 words have 16 occurrences, but only 5 of them fit in the top 25. Which ones are shown depends only on the order of appearance in the text, so positions 21–25 are partly arbitrary.

Adding the archaic forms and *er* to the stop list gives a list with only content words, where *infant*, *see*, *human* and *cloud* enter the top 25. This shows that the result depends on the stop list and on the tokenizer: a stop list must be adapted to the language variety of the text.